## Install Libraries

In [ ]:
!pip install transformers seqeval evaluate conllu accelerate datasets -q

# Import Libraries

In [ ]:
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from evaluate import load
from conllu import parse

# Download Universal Dependencies Dataset

In [ ]:
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu

# Load Dataset

In [ ]:
def load_ud_data(file_path):

    tokens_list = []
    pos_tags_list = []

    with open(file_path, "r", encoding="utf-8") as f:
        sentences = parse(f.read())

    for sentence in sentences:

        tokens = []
        pos_tags = []

        for token in sentence:
            tokens.append(token["form"])
            pos_tags.append(token["upos"])

        tokens_list.append(tokens)
        pos_tags_list.append(pos_tags)

    return tokens_list, pos_tags_list

# Prepare Train and Test Data

In [ ]:
train_tokens, train_pos = load_ud_data("en_ewt-ud-train.conllu")
test_tokens, test_pos = load_ud_data("en_ewt-ud-dev.conllu")

# Use only first 4000 samples
train_tokens = train_tokens[:4000]
train_pos = train_pos[:4000]

test_tokens = test_tokens[:500]
test_pos = test_pos[:500]

print(train_tokens[0])
print(train_pos[0])

# Create Label Mapping

In [ ]:
labels = sorted(list(set(tag for sent in train_pos for tag in sent)))

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

num_labels = len(labels)

print(labels)

# Load BERT Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenization + Label Alignment

In [ ]:
def tokenize_and_align(tokens, labels):

    tokenized = tokenizer(
        tokens,
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized.word_ids()

    label_ids = []
    prev = None

    for word_idx in word_ids:

        if word_idx is None:
            label_ids.append(-100)

        elif word_idx != prev:
            label_ids.append(label2id[labels[word_idx]])

        else:
            label_ids.append(-100)

        prev = word_idx

    tokenized["labels"] = label_ids

    return tokenized

# Preprocess

In [ ]:
train_encodings = [tokenize_and_align(t, l) for t,l in zip(train_tokens, train_pos)]
test_encodings = [tokenize_and_align(t, l) for t,l in zip(test_tokens, test_pos)]

# Convert to PyTorch Dataset

In [ ]:
class TokenDataset(torch.utils.data.Dataset):

    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):

        item = {k: torch.tensor(v) for k,v in self.encodings[idx].items()}
        return item

    def __len__(self):
        return len(self.encodings)


train_dataset = TokenDataset(train_encodings)
test_dataset = TokenDataset(test_encodings)

# Model Setup

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Data Collator

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

# Compute Metrics

In [ ]:
def compute_metrics(p):

    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    true_labels = [
        [id2label[l] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Training Configuration

In [ ]:
training_args = TrainingArguments(

    output_dir="./pos_model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100
)

# Trainer

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# Train model

In [ ]:
trainer.train()

# Evaluation Metric

In [ ]:
from evaluate import load

metric = load("seqeval")

# Evaluate

In [ ]:
trainer.evaluate()

# Inference 1

In [ ]:
sentence = "John works at Google in California"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

pred_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, pred_labels):
    print(token, label)

# Inference 2

In [ ]:
sentence = "im very good at coding i will be winning this hackathon"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

pred_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, pred_labels):
    print(token, label)

# Comparison POS tagging vs Chunking

| Feature    | POS Tagging                   | Chunking                 |
| ---------- | ----------------------------- | ------------------------ |
| Level      | Word-level                    | Phrase-level             |
| Goal       | Identify grammatical category | Identify phrases         |
| Example    | Noun, Verb, Adjective         | Noun Phrase, Verb Phrase |
| Difficulty | Easy                          | Medium                   |
| Dataset    | Universal Dependencies        | CoNLL-2000               |


Example

**Sentence:  The quick brown fox jumps over the lazy dog**

**POS** **tagging**:
The → DET,
quick → ADJ,
brown → ADJ,
fox → NOUN,
jumps → VERB,
over → ADP,
the → DET,
lazy → ADJ,
dog → NOUN

**Chunking**:
[The quick brown fox] → NP,
[jumps] → VP,
[over the lazy dog] → PP

**Chunking → Phrase-level grouping**

Chunking groups words into syntactic phrases such as noun phrases and verb phrases.
Unlike POS tagging which labels individual words, chunking identifies meaningful word groups.
This task is more complex because the model must learn relationships between neighboring tokens.